# A5 — assemble the tables

No GPU. No training. Nothing is written outside `protocol_v2/results/derived/`.

Reads five arms:

| arm | where from | trained here? |
|---|---|---|
| `P2_cutoff` | `protocol_v2/results/arms/P2_cutoff/` | yes, 200 epochs |
| `P1_causal` | `protocol_v2/results/arms/P1_causal/` | yes, 200 epochs |
| `P1_leaky` | `protocol_v2/results/arms/P1_leaky/` | yes, 200 epochs |
| `P3_acausal` | `protocol_v2/results/arms/P3_acausal/` | yes, 200 epochs |
| **`P3_rolling`** | `results/rerun_20260826_114717_m5/canonical/main/` | **no — read-only** |

Every analysis choice comes from `assembly_settings.py`, which was frozen before the first
arm was trained. Its SHA-256 is recorded in the manifest, so an edit after the results
were seen would show.

Two things to keep in mind while reading the output:

* **The five rungs come from two sessions.** Four are new; `P3_rolling` is the August run.
  Every setting is matched; the sitting is not. That is a real difference and the manifest
  records both.
* **AUROC is computed here, from the retained scores**, not inside the training loop as it
  was in the 75-epoch study. The number is the same; moving it out is what lets the
  training code stay identical to Section 5.1.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

PV2 = Path.cwd() if Path.cwd().name == "protocol_v2" else Path.cwd().parent

# protocol_v2 LAST so it lands FIRST — src/config.py would otherwise shadow ours.
for d in (str(PV2.parent / "src"), str(PV2.parent), str(PV2)):
    while d in sys.path:
        sys.path.remove(d)
    sys.path.insert(0, d)

import warnings; warnings.filterwarnings("ignore")
import json
import numpy as np, pandas as pd

import config as CFG
assert Path(CFG.__file__).parent == PV2, f"wrong config module: {CFG.__file__}"
import guards, provenance as prov, runner, splits as SP
import assembly_settings as ASET, assemble as ASM

# the canonical arm, and the run manifest one level above it, both unchanged
lock = guards.assert_canonical_unchanged()

# the analysis has not moved since it was frozen, before the first arm was trained
analysis_lock = prov.verify_analysis_lock()
print(f"             frozen at {analysis_lock['locked_utc']}")
print()
ASET.print_frozen()

## 1b. Refuse to assemble anything incomplete

This gate exists because without it a `P3_acausal` that had only run its two smoke folds would produce a two-week ladder and no warning — `ladder()` intersects whatever weeks it finds. A partial ladder is worse than no ladder, because it looks like a result.

In [ ]:
complete = ASM.assert_arms_complete()

## 2. The frozen rules, printed in full

Read these before looking at any number below.

In [ ]:
print("FOLD-SET RULE")
for k, v in ASET.FOLD_SET_RULE.items():
    print(f"  {k}: {v}")
print()
print("WILCOXON")
for k, v in ASET.WILCOXON.items():
    print(f"  {k}: {v}")
print()
print("CONTRASTS (pre-declared, reported as a - b)")
for name, a, b in ASET.CONTRASTS:
    print(f"  {name:38} {a} - {b}")
print()
print("NOT USED")
for k, v in ASET.DO_NOT_USE.items():
    print(f"  {k}\n     {v}")

## 3. Load the frame and the saved fold sets

Needed to recover the week of each scored row. `row_pos` in `scores.csv` is a per-fold counter — Section 5.1's schema, kept unchanged on purpose — so the global row comes from the saved test-index arrays instead. That is review item D7: identifiers in post-processing, not a parameter on the training function.

In [ ]:
B = runner.load_frame()
sp, week_of_row = B["splits"], B["week_of_row"]

## 4. Per-week metrics for all five arms

In [ ]:
pw = ASM.all_per_week(sp, week_of_row)
print()
ASM.assert_week_coverage(pw)
print()
print(pw.groupby("arm").agg(weeks=("test_week", "size"),
                            first=("test_week", "min"),
                            last=("test_week", "max"),
                            mean_ap=("ap", "mean")).reindex(list(ASET.LADDER_ORDER)))

## 5. Table 6 — the protocol ladder

In [ ]:
lad, common_weeks = ASM.ladder(pw)
print(f"weeks common to all five arms: {len(common_weeks)}  {common_weeks}")
print()
display(lad)
print()
print(f"AP range    {lad.ap.max() - lad.ap.min():.4f}")
print(f"AUROC range {lad.auroc.max() - lad.auroc.min():.4f}")

## 6. Table 7 — the decomposition

Note the `n_weeks` column: 16 where a P1 arm is involved, 17 otherwise. That is the fold-set rule, applied automatically and shown rather than assumed.

In [ ]:
dec = ASM.decomposition(pw, "ap")
display(dec[["effect", "contrast", "n_weeks", "weeks", "delta", "consistent",
             "p_value", "wilcoxon_method"]])

## 6b. The non-additive excess

**One quantity, one sign.** The aggregate excess *is* the mean of the per-week vector —
`non_additivity()` derives both from `assembly_settings.EXCESS_PER_WEEK` and asserts they
agree to 1e-12, then cross-checks against `total − (the two single corrections)`.

The convention, declared in `assembly_settings.SIGN_CONVENTION`, is the manuscript's:
corrections start from `P1_leaky`, the corner where both problems are present, and move
toward `P3_rolling`. The excess reduces to

```
P1_causal + P3_acausal − P1_leaky − P3_rolling
```

and a **positive** value means fixing either defect alone recovers much less than fixing
both.

Getting this wrong is not hypothetical. An earlier draft summed the two effects measured
*from* `P3_rolling` instead — 0.2301 rather than 0.0220 — which produced the right
magnitude with the opposite sign: −0.104 where the paper publishes +0.104. Note also that
`notebooks/08b_protocol_11feat.ipynb` is itself inconsistent here: its aggregate prints
+0.1040 and its per-week vector averages −0.1040. That inconsistency does not survive
into this package.

In [ ]:
na = ASM.non_additivity(pw, "ap")
for k, v in na["single_corrections"].items():
    print(f"  {k:34} {v:+.4f}")
print(f"  {'sum of the two':34} {na['sum_of_single_corrections']:+.4f}")
print(f"  {'total protocol effect':34} {na['total_protocol_effect']:+.4f}")
print(f"  {'NON-ADDITIVE EXCESS':34} {na['excess']:+.4f}")
print()
print(f"  formula      {na['formula']}")
print(f"  positive means {na['sign_means']}")
print(f"  per week     n={na['n']}, {na['consistent_weeks']}/{na['n']} weeks positive, "
      f"Wilcoxon p={na['p_value']:.5f} ({na['wilcoxon_method']})")
print()
print("  per-week vector:")
display(ASM.excess_per_week(pw).rename("excess").to_frame().T.round(4))

## 7. Figure 6 — the 2×2

In [ ]:
fac = ASM.factorial(pw)
display(fac)
inter = na          # same object; interaction() and non_additivity() are one quantity

## 8. Write the derived tables and the top-level manifest

In [ ]:
written = ASM.write_all(pw, lad, common_weeks, dec, fac, inter)
print(written["dir"])
for f in written["files"]:
    print("  ", f)

arm_summaries = {}
for arm in CFG.TRAINABLE_ARMS:
    p = CFG.ARMS_DIR / arm / "summary.json"
    arm_summaries[arm] = json.loads(p.read_text()) if p.exists() else None

top = {
    "study": "TRACE-GNN protocol study, version 2",
    "what_changed": "All five arms now share the canonical 200-epoch configuration. The "
                    "published study trained at 75, and because CosineAnnealingLR anneals "
                    "over T_max=epochs, 75 and 200 are different configurations rather "
                    "than a short and a long version of one.",
    "arms_trained_here": list(CFG.TRAINABLE_ARMS),
    "arms_reused_untrained": list(CFG.LOCKED_ARMS),
    "canonical_p3_lock": lock,
    "provenance_split": ASET.PROVENANCE_SPLIT,
    "sign_convention": ASET.SIGN_CONVENTION,
    "analysis_lock": analysis_lock,
    "arm_completeness": {k: (v.get("summary") or v) for k, v in complete.items()},
    "frozen_analysis_settings_sha256": ASET.settings_sha256(),
    "splits_manifest": json.loads(Path(SP.MANIFEST).read_text()),
    "source_hashes": prov.source_hashes(),
    "environment": prov.environment_manifest(
        B["prepared"] / "sparkov_clean_v1.csv"),
    "environment_vs_canonical": prov.compare_environment(
        prov.environment_manifest(B["prepared"] / "sparkov_clean_v1.csv"), verbose=False),
    "arm_summaries": arm_summaries,
    "ladder_weeks": common_weeks,
    "deprecated": {"src/protocol_runner.py":
                   "stale copy of the training loop; not used by protocol_v2"},
}
prov.write_manifest(CFG.RESULTS / "run_manifest.json", top)
print()
print("wrote", CFG.RESULTS / "run_manifest.json")

## 9. Final lock check

In [ ]:
after = guards.assert_canonical_unchanged()
assert after["combined_sha256"] == lock["combined_sha256"]
print("canonical P3_rolling untouched throughout assembly")

---

## Cleanup

This notebook uses no GPU, so there is nothing to free. It does hold the 500,000-row
frame in memory, so shut the kernel down when you are finished with it.

**What to check before touching the manuscript**

1. `results/derived/protocol_ladder.csv` — five rows, all on the same weeks.
2. `results/derived/leak_decomposition.csv` — the `n_weeks` column reads 16 or 17 exactly
   as the fold-set rule says it should.
3. `results/run_manifest.json` — `arms_reused_untrained` lists `P3_rolling`, and
   `canonical_p3_lock.combined_sha256` matches `results/canonical_p3_lock.json`.
4. `frozen_analysis_settings_sha256` matches `assembly_settings.settings_sha256()`.